In [ ]:
"""
Pulls nationwide (all-store) daily sales for FOODS_3_305, merges in calendar (events, SNAP) and price data, and writes a single clean CSV
that both the Prophet and XGBoost scripts will read from.

Run this FIRST, before either modeling script.

Requires (same folder as this script):
  - sales_train_evaluation.csv
  - sell_prices.csv
  - calendar.csv
"""

In [ ]:
import pandas as pd

ITEM_ID = "FOODS_3_305"

# Load Data
sales = pd.read_csv("sales_train_evaluation.csv")
calendar = pd.read_csv("calendar.csv")
prices = pd.read_csv("sell_prices.csv")

In [ ]:
# Aggregate daily sales for FOODS_3_305 across all stores

item_rows = sales[sales["item_id"] == ITEM_ID]
day_cols = [c for c in sales.columns if c.startswith("d_")]

daily_totals = item_rows[day_cols].sum(axis=0)  # sum across all 10 stores
daily_df = daily_totals.reset_index()
daily_df.columns = ["d", "units_sold"]


In [ ]:
# Merge calendar info (actual date, events, SNAP flags)

cal_cols = [
    "d", "date", "wm_yr_wk", "weekday", "wday", "month", "year",
    "event_name_1", "event_type_1", "event_name_2", "event_type_2",
    "snap_CA", "snap_TX", "snap_WI",
]
daily_df = daily_df.merge(calendar[cal_cols], on="d")
daily_df["date"] = pd.to_datetime(daily_df["date"])

# Simple flags
daily_df["has_event"] = daily_df["event_name_1"].notna().astype(int)
daily_df["any_snap"] = (
    (daily_df["snap_CA"] == 1) | (daily_df["snap_TX"] == 1) | (daily_df["snap_WI"] == 1)
).astype(int)


In [ ]:
# Merge price info, averaged across stores per week 

item_prices = prices[prices["item_id"] == ITEM_ID]
weekly_avg_price = (
    item_prices.groupby("wm_yr_wk")["sell_price"].mean().reset_index()
    .rename(columns={"sell_price": "avg_price"})
)
daily_df = daily_df.merge(weekly_avg_price, on="wm_yr_wk", how="left")

# Forward fill any missing price weeks (basically item not sold yet / gaps)
daily_df["avg_price"] = daily_df["avg_price"].ffill().bfill()


In [ ]:
# 5. Final clean and sva

final_cols = [
    "date", "units_sold", "avg_price", "weekday", "wday", "month", "year",
    "has_event", "event_name_1", "event_type_1", "any_snap",
]
daily_df = daily_df[final_cols].sort_values("date").reset_index(drop=True)

daily_df.to_csv("foods_3_305_daily.csv", index=False)
print(f"Saved foods_3_305_daily.csv with {len(daily_df)} rows")
print(daily_df.head())
print(daily_df.tail())